# 人工框解碼與 NMS

[完整圖文教材](https://birdhackor.github.io/learn_to_yolo/lessons/06-decode-nms/)

先執行環境格，再閱讀、執行下方完整實驗。各節互相獨立，不需要上一節的 runtime。

本節在 CPU 逐節執行並保存輸出；合成資料短訓練、L4 與部署紀錄見網站驗證頁。真實場景長訓練仍待安排。這些範例沒有預訓練權重，也不需下載資料。

In [ ]:
# 全新 runtime 可由此格開始。只取得小型程式，不自動下載 LFS 資料。
import importlib.metadata
import os
from pathlib import Path
import subprocess
import sys

REF = 'lessons-v0.3.0'
base = Path("/content") if Path("/content").is_dir() else Path.cwd()
repository = base / ("learn_to_yolo_" + REF.replace(".", "_").replace("-", "_"))
if not repository.exists():
    environment = os.environ.copy()
    environment["GIT_LFS_SKIP_SMUDGE"] = "1"
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REF,
                    "https://github.com/birdhackor/learn_to_yolo.git", str(repository)],
                   env=environment, check=True)
actual_ref = subprocess.check_output(
    ["git", "-C", str(repository), "describe", "--tags", "--exact-match"], text=True
).strip()
if actual_ref != REF:
    raise RuntimeError("程式版本不同，請使用新的 runtime 或移除舊 clone 後重試。")

# 逐節範例預設 CPU；L4 訓練與部署的實測另見網站驗證頁。保留相同版本的 CPU/CUDA build。
try:
    torch_version = importlib.metadata.version("torch").split("+")[0]
except importlib.metadata.PackageNotFoundError:
    torch_version = None
if torch_version != "2.9.1":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.9.1",
                    "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)
    if "torch" in sys.modules:
        raise RuntimeError("PyTorch 已更新：請從選單重新啟動工作階段，再由第一格執行。")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                *['numpy==2.3.5', 'pillow==12.0.0', 'matplotlib==3.10.7']], check=True)
os.chdir(repository)
sys.path.insert(0, str(repository))
print("固定教材版本：", REF)
print("目前目錄：", repository)


## 本節可修改的完整實驗

先預測結果，再執行；確認輸出後，試做網頁的自主練習。

In [1]:
"""Artificial logits make decoding/filtering/NMS independently checkable."""
import torch


def iou(box, boxes):
    left = torch.maximum(box[:2], boxes[:, :2])
    right = torch.minimum(box[2:], boxes[:, 2:])
    intersection = (right - left).clamp(min=0).prod(1)
    area = (box[2:] - box[:2]).clamp(min=0).prod()
    areas = (boxes[:, 2:] - boxes[:, :2]).clamp(min=0).prod(1)
    return intersection / (area + areas - intersection).clamp(min=1e-8)


def class_nms(boxes, scores, labels, threshold=0.5):
    keep = []
    for label in labels.unique():
        indices = (labels == label).nonzero().flatten()
        order = indices[scores[indices].argsort(descending=True, stable=True)]
        while order.numel():
            current = order[0]
            keep.append(current.item())
            remaining = order[1:]
            order = remaining[iou(boxes[current], boxes[remaining]) <= threshold]
    kept = torch.tensor(keep, dtype=torch.long)
    return kept[scores[kept].argsort(descending=True, stable=True)]


def decode(logits, score_threshold=0.25, image_size=64):
    grid = logits.shape[0]
    row, col = torch.meshgrid(torch.arange(grid), torch.arange(grid), indexing="ij")
    center = (torch.stack((col, row), -1) + logits[..., :2].sigmoid()) / grid * image_size
    size = logits[..., 2:4].sigmoid() * image_size
    boxes = torch.cat((center - size / 2, center + size / 2), -1)
    class_prob, labels = logits[..., 5:].softmax(-1).max(-1)
    scores = logits[..., 4].sigmoid() * class_prob
    selected = scores >= score_threshold
    return boxes[selected], scores[selected], labels[selected]


def fill(logits, row, col, xywh, obj, class_prob):
    logits[row, col, :4] = torch.logit(torch.tensor(xywh))
    logits[row, col, 4] = torch.logit(torch.tensor(obj))
    logits[row, col, 5:] = torch.tensor(class_prob).log()


def main():
    torch.manual_seed(7)
    torch.set_num_threads(2)
    logits = torch.zeros(4, 4, 7)
    logits[..., 4] = -12
    fill(logits, 1, 2, [.25, .75, .25, .125], .9, [.8, .2])
    fill(logits, 1, 1, [.95, .75, .25, .125], .8, [.8, .2])
    fill(logits, 3, 0, [.5, .5, .125, .125], .95, [.9, .1])
    boxes, scores, labels = decode(logits)
    assert len(boxes) == 3
    assert torch.allclose(boxes[1], torch.tensor([28., 24., 44., 32.]), atol=1e-5)
    assert torch.allclose(scores, torch.tensor([.64, .72, .855]), atol=1e-6)
    overlap = iou(boxes[1], boxes[:1]).item()
    assert abs(overlap - 7 / 13) < 1e-5
    kept = class_nms(boxes, scores, labels)
    assert kept.tolist() == [2, 1]
    print(f"candidate_boxes={boxes.tolist()}")
    print(f"scores={[round(s.item(), 3) for s in scores]}, duplicate_IoU={overlap:.4f}")
    print(f"NMS keep_indices={kept.tolist()}, kept_scores={[round(s.item(), 3) for s in scores[kept]]}")
    strict_boxes, strict_scores, _ = decode(logits, score_threshold=.75)
    assert len(strict_boxes) == 1 and torch.allclose(strict_boxes[0], boxes[2])
    print(f"threshold .75 keeps only artificial false positive, score={strict_scores.item():.3f}")
    # Independent exercises: preserve the baseline arrays, thresholds, and assertions.
    keep06 = class_nms(boxes, scores, labels, threshold=.6)
    assert keep06.tolist() == [2, 1, 0]
    print(f"exercise NMS .6 keeps original candidate indices={keep06.tolist()}")
    boxes70, scores70, labels70 = decode(logits, score_threshold=.70)
    assert len(boxes70) == 2 and labels70.tolist() == [0, 0]
    assert torch.allclose(scores70, torch.tensor([.72, .855]), atol=1e-6)
    print(f"exercise score .70 keeps scores={[round(s.item(), 3) for s in scores70]}")
    same_boxes = torch.stack((boxes[1], boxes[1]))
    class_keep = class_nms(same_boxes, torch.tensor([.9, .8]), torch.tensor([0, 1]))
    assert class_keep.tolist() == [0, 1]
    assert class_nms(torch.empty(0, 4), torch.empty(0), torch.empty(0, dtype=torch.long)).numel() == 0
    print("identical boxes of different classes both survive class-wise NMS; empty input passed")
    print("All logits are hand-constructed, not a trained detector.")


if __name__ == "__main__":
    main()


candidate_boxes=[[23.200000762939453, 24.0, 39.20000076293945, 32.0], [28.0, 24.0, 44.0, 32.0], [3.999999523162842, 52.0, 12.0, 60.0]]
scores=[0.64, 0.72, 0.855], duplicate_IoU=0.5385
NMS keep_indices=[2, 1], kept_scores=[0.855, 0.72]
threshold .75 keeps only artificial false positive, score=0.855
exercise NMS .6 keeps original candidate indices=[2, 1, 0]
exercise score .70 keeps scores=[0.72, 0.855]
identical boxes of different classes both survive class-wise NMS; empty input passed
All logits are hand-constructed, not a trained detector.
